#Step-by-step guide to implementing neural networks
Step 1: Set up an Azure Machine Learning workspace
Before implementing a neural network, we need to set up an Azure Machine Learning workspace. You can do this by following these steps:

In [ ]:
# Install the Azure Machine Learning SDK
#pip install azureml-core

#Next, create a new workspace or use an existing one:

In [2]:
from azureml.core import Workspace

# Create or retrieve an existing Azure ML workspace
#ws = Workspace.get(
#    name='myworkspace',
#    subscription_id='be278ba0-abd2-43b6-b02f-18580dddd5a7',
#    resource_group='mthabisi-rg',
#    location='East US 2')

ws = Workspace.create(
    name='myworkspace',
    subscription_id='be278ba0-abd2-43b6-b02f-18580dddd5a7',
    resource_group='mthabisi-rg',
    location='East US 2')

# Write configuration to the workspace config file
ws.write_config(path='.azureml')

Performing interactive authentication. Please follow the instructions on the terminal.


Interactive authentication successfully completed.
Deploying StorageAccount with name myworkspstorage734cb173d.
Deploying KeyVault with name myworkspkeyvault48121099.
Deploying AppInsights with name myworkspinsights9b5683c6.
Deployed AppInsights with name myworkspinsights9b5683c6. Took 1.74 seconds.
Deployed StorageAccount with name myworkspstorage734cb173d. Took 20.54 seconds.
Deployed KeyVault with name myworkspkeyvault48121099. Took 18.26 seconds.
Deploying Workspace with name myworkspace.
Deployed Workspace with name myworkspace. Took 35.86 seconds.


#Step 2: Build a simple neural network using PyTorch
Now that we have our workspace set up, let’s define a simple neural network. PyTorch is well-suited for building neural networks, and it integrates seamlessly with Azure.

Here's a basic implementation of a feedforward neural network:

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define a simple neural network with one hidden layer
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(784, 128)  # Input layer (784 input features)
        self.fc2 = nn.Linear(128, 10)   # Output layer (10 classes)
        self.relu = nn.ReLU()           # Activation function

    def forward(self, x):
        x = self.relu(self.fc1(x))  # Apply ReLU after the first layer
        x = self.fc2(x)             # Output layer
        return x

# Initialize the neural network
model = SimpleNN()

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # For classification tasks
optimizer = optim.SGD(model.parameters(), lr=0.01)  # Stochastic Gradient Descent

In the above code, we:

Defined a simple neural network with an input layer, one hidden layer, and an output layer.

Used the ReLU (rectified linear unit) activation function, which is commonly used in neural networks.

Defined a CrossEntropyLoss for multi-class classification and SGD (stochastic gradient descent) as the optimizer.

#Step 3: Train the neural network
After defining the model, we need to train it on a dataset. We will simulate training using the MNIST dataset (a set of handwritten digits) to demonstrate the process. First, we load the data and then train the model.

In [4]:

from torchvision import datasets, transforms

# Load MNIST dataset
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(root='data', train=True, download=True,
                   transform=transforms.ToTensor()),
    batch_size=32, shuffle=True)

# Train the model
num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()  # Reset gradients

        # Forward pass
        outputs = model(inputs.view(-1, 784))  # Flatten input
        loss = criterion(outputs, labels)      # Compute loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.56MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.08MB/s]


Epoch 1, Loss: 0.866190177377065
Epoch 2, Loss: 0.37456730968554813
Epoch 3, Loss: 0.31843378201325734
Epoch 4, Loss: 0.2874812227110068
Epoch 5, Loss: 0.26416012436946235


In this code:

The MNIST dataset is loaded using torchvision.

The network is trained for five epochs using stochastic gradient descent to update the weights.

The input is flattened (from 28 × 28 images to 784 features) before being fed into the network.

#Step 4: Deploy the model on Azure
After training the model, you can deploy it to Azure Machine Learning for inference at scale. Here's how to register the model and deploy it as a web service:

In [5]:
from azureml.core import Model

# Save the trained model
torch.save(model.state_dict(), 'simple_nn.pth')

# Register the model in Azure
model = Model.register(workspace=ws, model_path='simple_nn.pth', model_name='simple_nn')

# Deploying as a web service requires more steps such as creating a scoring script and environment.

Registering model simple_nn


In [7]:
from azureml.core import Workspace, Model
import torch
import os

# Check current directory
print("Current directory:", os.getcwd())

# Verify model file exists
print("File exists:", os.path.exists("simple_nn.pth"))

# Connect to workspace
ws = Workspace.from_config()

print("Workspace:", ws.name)

# Register model
registered_model = Model.register(
    workspace=ws,
    model_path="simple_nn.pth",
    model_name="simple_nn"
)

print("Registered model name:", registered_model.name)
print("Version:", registered_model.version)

Current directory: /content
File exists: True
Workspace: myworkspace
Registering model simple_nn
Registered model name: simple_nn
Version: 2
